# 4.6 IC Deriving New Variables

In this notebook, I create new variables in the `ords_prods_merge` dataframe, including `price_label`, `busiest_day`, `busiest_days`, and `busiest_period_of_day`, following the task requirements.

In [2]:
# 01. Import libraries and data
import pandas as pd
import os

# Set base path
path = r"/Users/a/Documents/Instacart Basket Analysis"

# Path to merged pickle file
merge_path = os.path.join(path, "02 Data", "Prepared Data", "ords_prods_merge.pkl")

# Import dataframe
df_ords_prods = pd.read_pickle(merge_path)

# Quick preview
df_ords_prods.head(), df_ords_prods.shape

(   order_id  user_id  order_number  order_dow  order_hour_of_day  \
 0   2539329        1             1          2                  8   
 1   2539329        1             1          2                  8   
 2   2539329        1             1          2                  8   
 3   2539329        1             1          2                  8   
 4   2539329        1             1          2                  8   
 
    days_since_prior_order  product_id  add_to_cart_order  reordered  \
 0                     0.0         196                  1          0   
 1                     0.0       14084                  2          0   
 2                     0.0       12427                  3          0   
 3                     0.0       26088                  4          0   
 4                     0.0       26405                  5          0   
 
                               product_name  aisle_id  department_id  prices  
 0                                     Soda      77.0            7.0   

In [4]:
# 02. Create price_label column
df_ords_prods.loc[df_ords_prods['prices'] <= 5, 'price_label'] = 'Low'
df_ords_prods.loc[(df_ords_prods['prices'] > 5) & (df_ords_prods['prices'] <= 15), 'price_label'] = 'Mid'
df_ords_prods.loc[df_ords_prods['prices'] > 15, 'price_label'] = 'High'

# Quick check
df_ords_prods['price_label'].value_counts()

price_label
Mid     21890146
Low     10126384
High      417682
nan         2029
Name: count, dtype: int64

In [6]:
# 03. Create busiest_day column

# Frequency of orders by day of week
ords_freq = df_ords_prods['order_dow'].value_counts(normalize=True)

# Create busiest_day labels
df_ords_prods.loc[df_ords_prods['order_dow'].isin(ords_freq[ords_freq > ords_freq.mean()].index), 'busiest_day'] = 'Busy'
df_ords_prods.loc[df_ords_prods['order_dow'].isin(ords_freq[ords_freq < ords_freq.mean()].index), 'busiest_day'] = 'Slow'
df_ords_prods['busiest_day'].fillna('Average', inplace=True)

# Quick check
df_ords_prods['busiest_day'].value_counts()

busiest_day
Slow    20560034
Busy    11876207
Name: count, dtype: int64

In [8]:
# 04. Create busiest_days (plural) column based on top 2 busiest and bottom 2 slowest days

# Calculate frequency of orders by day of week
dow_counts = df_ords_prods['order_dow'].value_counts()

# Identify two busiest days and two slowest days
top2 = dow_counts.nlargest(2).index
bottom2 = dow_counts.nsmallest(2).index

# Create new column
df_ords_prods.loc[df_ords_prods['order_dow'].isin(top2), 'busiest_days'] = 'Busiest'
df_ords_prods.loc[df_ords_prods['order_dow'].isin(bottom2), 'busiest_days'] = 'Slowest'

# Everything else = 'Average'
df_ords_prods['busiest_days'].fillna('Average', inplace=True)

# Quick check
df_ords_prods['busiest_days'].value_counts()

busiest_days
nan        12928278
Busiest    11876207
Slowest     7631756
Name: count, dtype: int64

In [10]:
# 04 (fix). Recreate busiest_days so every row is labeled

# Frequency of orders by day of week
dow_counts = df_ords_prods['order_dow'].value_counts()

# Two busiest and two slowest days
top2 = dow_counts.nlargest(2).index
bottom2 = dow_counts.nsmallest(2).index

# Start by labeling everything as 'Average'
df_ords_prods['busiest_days'] = 'Average'

# Overwrite with Busiest and Slowest where appropriate
df_ords_prods.loc[df_ords_prods['order_dow'].isin(top2), 'busiest_days'] = 'Busiest'
df_ords_prods.loc[df_ords_prods['order_dow'].isin(bottom2), 'busiest_days'] = 'Slowest'

# Quick check
df_ords_prods['busiest_days'].value_counts()

busiest_days
Average    12928278
Busiest    11876207
Slowest     7631756
Name: count, dtype: int64

In [12]:
# 05. Create busiest_period_of_day column

# Frequency of orders by hour of day
hour_counts = df_ords_prods['order_hour_of_day'].value_counts().sort_index()

# Thresholds for "Fewest", "Average", "Most" based on distribution
low_threshold = hour_counts.quantile(1/3)
high_threshold = hour_counts.quantile(2/3)

# Function to label each hour
def label_period(hour):
    count = hour_counts[hour]
    if count <= low_threshold:
        return 'Fewest orders'
    elif count >= high_threshold:
        return 'Most orders'
    else:
        return 'Average orders'

# Create new column
df_ords_prods['busiest_period_of_day'] = df_ords_prods['order_hour_of_day'].apply(label_period)

# Quick check: frequency of the new column
df_ords_prods['busiest_period_of_day'].value_counts()

busiest_period_of_day
Most orders       21138591
Average orders    10007334
Fewest orders      1290316
Name: count, dtype: int64

In [14]:
df_ords_prods['busiest_period_of_day'].value_counts()

busiest_period_of_day
Most orders       21138591
Average orders    10007334
Fewest orders      1290316
Name: count, dtype: int64

### Checks on new derived variables

**busiest_days**

- I created `busiest_days` using the two most frequent days (labeled **“Busiest”**) and the two least frequent days (labeled **“Slowest”**).
- All remaining days were labeled **“Average”**.
- The value counts confirm that every row has a label and that “Busiest” and “Slowest” represent smaller subsets compared to “Average”, which matches the intended logic.

**busiest_period_of_day**

- I calculated order frequencies by `order_hour_of_day` and split them into three groups using the 1/3 and 2/3 quantiles.
- Hours below the lower threshold are labeled **“Fewest orders”**, above the upper threshold **“Most orders”**, and the rest **“Average orders”**.
- The value counts show all rows labeled and a reasonable distribution across the three categories, so the column works as expected.

In [17]:
# 7. Export updated dataframe with new derived variables
export_path = os.path.join(path, "02 Data", "Prepared Data", "ords_prods_derived.pkl")
df_ords_prods.to_pickle(export_path)

export_path

'/Users/a/Documents/Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_derived.pkl'